# Feature Selection

### Setup

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
# Setup working directory
from pathlib import Path
import os

def find_root_dir(marker='tfg'):
    p = Path.cwd()
    for candidate in [p] + list(p.parents):
        if candidate.name == marker:
            return candidate.resolve()
        if (candidate / marker).is_dir():
            return (candidate / marker).resolve()
    raise FileNotFoundError(f"Could not find '{marker}' folder in {Path.cwd()} or its parents.")

os.chdir(find_root_dir('tfg'))

## Load Data

In [3]:
FULL_DATASET_PATH = "data/processed/stage2/full-225k-baseline.parquet"

In [4]:
from src.preprocessing import load_dataset
df = load_dataset(FULL_DATASET_PATH)
print("\nFirst few rows:")
df.head()


First few rows:


,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,Total Backward Packets,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,92.168.10.5-104.16.207.165-54865-4143-6,104.16.207.165,443,192.168.10.5,54865,6,7/7/2017 3:30,3,2,0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,192.168.10.5-104.16.28.216-55054-80-6,104.16.28.216,80,192.168.10.5,55054,6,7/7/2017 3:30,109,1,1,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,192.168.10.5-104.16.28.216-55055-80-6,104.16.28.216,80,192.168.10.5,55055,6,7/7/2017 3:30,52,1,1,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,192.168.10.16-104.17.241.25-46236-443-6,104.17.241.25,443,192.168.10.16,46236,6,7/7/2017 3:30,34,1,1,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,192.168.10.5-104.19.196.102-54863-443-6,104.19.196.102,443,192.168.10.5,54863,6,7/7/2017 3:30,3,2,0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


## Structural Filtering

The **categorical** features are excluded, as the Stage I results showed that their inclusion provided only a marginal performance improvement while increasing the dimensionality and complexity of the feature space. Keep `Timestamp` for the temporal split. In addition, duplicated features, near-zero variance features, and the TCP Window features are removed. The latter are excluded because the presence of negative values violates the expected logical constraints of TCP window measurements and may reflect inconsistencies during feature extraction. 

This structural filtering establishes a consistent set of informative features that is subsequently used for the experimental feature selection and modelling stages.

---

In [5]:
categorical_to_exclude = ['Flow ID', 'Source IP', 'Destination IP', 'Timestamp', 'Source Port', 'Destination Port', 'Protocol']
tcp_flags_to_exclude = ['Fwd PSH Flags','FIN Flag Count','RST Flag Count','ECE Flag Count']
window_to_exclude = ['Init_Win_bytes_forward', 'Init_Win_bytes_backward']
active_idle_to_exclude = ['Active Std', 'Idle Std']

In [6]:
features_to_exclude = (
    tcp_flags_to_exclude
    + active_idle_to_exclude
    + window_to_exclude
    + categorical_to_exclude
)

In [7]:
df_clean = df.drop(columns=features_to_exclude)

## Save Datasets

In [8]:
from src.preprocessing import save
save("stage2/full-225k-filtered.parquet", df_clean)

Saving dataset as .parquet
Final Dataset Saved: data/processed\stage2/full-225k-filtered.parquet (23.91 MB) 
